# 02. Data Preparation

This notebook will be used to prepare the input data for the ClearCNV model. We will load the `SpatialData` object, explore it, and then prepare the three required inputs for the model:
1. Gene expression matrix
2. Spatial graph
3. Gene bins

In [1]:
import os
import spatialdata as sd
import numpy as np
import geopandas as gpd
from sklearn.neighbors import kneighbors_graph
import torch
import infercnvpy as cnv
import scanpy as sc
import scipy.sparse

/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  return _bootstrap._gcd_import(name[level:], package, level)
/usr/local/lib/python3.12/dist-packages/cudf/utils/gpu_utils.py:75: UserWarning: Failed to dlopen libcuda.so.1
  warnings.warn(str(e))


## 1. Load the SpatialData object

In [2]:
data_dir = "../datasets"
sdata_path = os.path.join(data_dir, "Xenium5K_human_prostate.zarr")
sdata = sd.read_zarr(sdata_path)

/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: Use

In [3]:
sdata

SpatialData object, with associated Zarr store: /content/drive/MyDrive/Courses/DDLS_course/Final_Project/clearCNV/datasets/Xenium5K_human_prostate.zarr
├── Images
│     └── 'morphology_focus': DataTree[cyx] (4, 30420, 54160), (4, 15210, 27080), (4, 7605, 13540), (4, 3802, 6770), (4, 1901, 3385)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (30420, 54160), (15210, 27080), (7605, 13540), (3802, 6770), (1901, 3385)
│     └── 'nucleus_labels': DataTree[yx] (30420, 54160), (15210, 27080), (7605, 13540), (3802, 6770), (1901, 3385)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (193000, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (193000, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (184853, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (193000, 5006)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), cell_l

## 2. Explore the SpatialData object

Now that we have loaded the `SpatialData` object, we can explore its contents. A `SpatialData` object can contain multiple elements, such as images, tables, and shapes.

In [4]:
print(f"Images: {list(sdata.images.keys())}")
print(f"Tables: {list(sdata.tables.keys())}")
print(f"Shapes: {list(sdata.shapes.keys())}")

Images: ['morphology_focus']
Tables: ['table']
Shapes: ['cell_boundaries', 'cell_circles', 'nucleus_boundaries']


### 2.1. Gene Expression Matrix

The gene expression data is stored in the `table` element. We can access it and convert it to a torch tensor.

In [5]:
sdata.tables['table'].layers['raw'] = sdata.tables['table'].X.copy()

In [6]:
sc.pp.normalize_total(sdata.tables['table'])
sc.pp.log1p(sdata.tables['table'])

/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:82: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


In [7]:
print(sdata.tables['table'].X)

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 30158302 stored elements and shape (193000, 5006)>
  Coords	Values
  (0, 1267)	4.969813346862793
  (1, 31)	0.35020241141319275
  (1, 38)	0.35020241141319275
  (1, 57)	0.35020241141319275
  (1, 72)	0.9848533868789673
  (1, 79)	0.6090640425682068
  (1, 98)	0.35020241141319275
  (1, 154)	0.35020241141319275
  (1, 185)	0.35020241141319275
  (1, 194)	0.35020241141319275
  (1, 208)	0.35020241141319275
  (1, 211)	0.35020241141319275
  (1, 213)	0.35020241141319275
  (1, 273)	0.35020241141319275
  (1, 297)	0.35020241141319275
  (1, 301)	0.6090640425682068
  (1, 303)	0.35020241141319275
  (1, 307)	0.35020241141319275
  (1, 320)	0.35020241141319275
  (1, 323)	0.35020241141319275
  (1, 358)	0.35020241141319275
  (1, 371)	0.35020241141319275
  (1, 374)	0.35020241141319275
  (1, 385)	0.35020241141319275
  (1, 393)	0.35020241141319275
  :	:
  (192999, 2510)	1.195875644683838
  (192999, 2541)	1.195875644683838
  (192999, 2544)	1.19587564468

In [8]:
expression_matrix = sdata.tables['table'].X.toarray()
print(f"Expression matrix shape: {expression_matrix.shape}")

Expression matrix shape: (193000, 5006)


### 2.2. Spatial Graph

To build the spatial graph, we need the spatial coordinates of the cells. These are stored in the `shapes` element.

In [9]:
# Extract cell centroids from cell boundaries
centroids = sdata.shapes["cell_boundaries"].geometry.centroid

# Make a dataframe with cell_id and x/y coordinates
cell_coords = gpd.GeoDataFrame({
    "x": centroids.x,
    "y": centroids.y
})

cell_coords

,x,y
aaaagkdm-1,170.746566,2017.138568
aaaamcnn-1,141.483477,2481.337132
aaaamjle-1,198.248333,2414.409944
aaabjaeg-1,129.224632,2834.541407
aaablgce-1,649.541930,2995.561032
...,...,...
oioggaih-1,4891.633158,3019.939870
oiogjhoo-1,4852.881563,2971.802085
oiogldij-1,4860.883555,2969.689216
oioglipl-1,4906.423643,3140.026792


In [10]:
# --- Build the k-NN graph (this returns a scipy.sparse.csr_matrix) ---
n_neighbors = 6
spatial_graph_sparse_scipy = kneighbors_graph(cell_coords, n_neighbors=n_neighbors, mode='connectivity', include_self=False)

print("Built k-NN graph in scipy sparse format.")

# --- Convert the scipy sparse matrix to a PyTorch sparse tensor ---

# 1. Get the matrix in COOrdinate format, which is easy to work with
coo = spatial_graph_sparse_scipy.tocoo()

# 2. Create the indices for the sparse tensor (the coordinates of the non-zero values)
indices = torch.from_numpy(np.vstack((coo.row, coo.col))).long()

# 3. Create the values for the sparse tensor (the non-zero values themselves)
values = torch.from_numpy(coo.data).float()

# 4. Get the shape of the original matrix
shape = torch.Size(coo.shape)

# 5. Create the PyTorch sparse tensor
spatial_graph_sparse_tensor = torch.sparse_coo_tensor(indices, values, shape)

Built k-NN graph in scipy sparse format.


In [11]:
spatial_graph_sparse_tensor

tensor(indices=tensor([[     0,      0,      0,  ..., 192999, 192999, 192999],
                       [ 85280,  85281,  85282,  ..., 186975, 186966, 186959]]),
       values=tensor([1., 1., 1.,  ..., 1., 1., 1.]),
       size=(193000, 193000), nnz=1158000, layout=torch.sparse_coo)

Now we can build a spatial graph. A common approach is to use a k-nearest neighbors (k-NN) graph.

### 2.3. Gene Bins

To create the gene bins, we need information about the genomic location of each gene. This information is often stored in the `sdata.table.var` dataframe.

In [13]:
sdata.tables['table'].var.head()

,gene_ids,feature_types,genome
A2ML1,ENSG00000166535,Gene Expression,Unknown
AAMP,ENSG00000127837,Gene Expression,Unknown
AAR2,ENSG00000131043,Gene Expression,Unknown
AARSD1,ENSG00000266967,Gene Expression,Unknown
ABAT,ENSG00000183044,Gene Expression,Unknown


In [14]:
cnv.io.genomic_position_from_biomart(sdata.tables['table'], adata_gene_id="gene_ids" ,species="hsapiens")

In [15]:
sdata.tables['table'].var.head()

,gene_ids,feature_types,genome,ensembl_gene_id,start,end,chromosome
A2ML1,ENSG00000166535,Gene Expression,Unknown,ENSG00000166535,8822621,8887001,chr12
AAMP,ENSG00000127837,Gene Expression,Unknown,ENSG00000127837,218264125,218270178,chr2
AAR2,ENSG00000131043,Gene Expression,Unknown,ENSG00000131043,36236131,36270918,chr20
AARSD1,ENSG00000266967,Gene Expression,Unknown,ENSG00000266967,42950431,42964498,chr17
ABAT,ENSG00000183044,Gene Expression,Unknown,ENSG00000183044,8674596,8784575,chr16


Now we need to define a strategy to bin the genes. For example, we can bin them by chromosome.

In [16]:
import pandas as pd

def create_gene_bins(gene_info_df, genes_per_bin=100):
    """
    Creates genomic bins with a fixed number of genes.

    Args:
        gene_info_df (pd.DataFrame): DataFrame with gene information, must contain 'chromosome', 'start', and 'end' columns.
        genes_per_bin (int): The number of genes to include in each bin.

    Returns:
        dict: A dictionary where keys are bin names (e.g., 'chr1_bin1') and
              values are lists of integer indices of the genes in that bin.
    """

    # --- 1. Prepare and sort the gene information ---

    # Make a copy to avoid modifying the original dataframe
    gene_info = gene_info_df.copy()

    # Drop genes with no chromosome information
    gene_info = gene_info.dropna(subset=['chromosome', 'start'])

    # Ensure chromosome names are strings and sort them naturally (e.g., chr1, chr2, ..., chrX)
    gene_info['chromosome'] = gene_info['chromosome'].astype(str)

    # Create a categorical type for natural sorting of chromosome names
    chrom_order = [f'chr{i}' for i in range(1, 23)] + ['chrX', 'chrY']
    gene_info['chromosome'] = pd.Categorical(gene_info['chromosome'], categories=chrom_order, ordered=True)

    # Sort genes by chromosome and start position
    sorted_genes = gene_info.sort_values(['chromosome', 'start'])

    # --- 2. Create the bins ---

    gene_bins = {}
    current_chrom = None
    bin_counter = 0

    for i in range(0, len(sorted_genes), genes_per_bin):
        bin_df = sorted_genes.iloc[i:i+genes_per_bin]

        # Get the chromosome of the first gene in the bin
        chrom = bin_df['chromosome'].iloc[0]

        # Reset bin counter for each new chromosome
        if chrom != current_chrom:
            bin_counter = 0
            current_chrom = chrom

        bin_name = f"{chrom}_bin{bin_counter}"

        # Get the integer indices of the genes in the bin
        # These indices correspond to the rows in the original expression matrix
        gene_indices = bin_df.index.map(lambda x: gene_info_df.index.get_loc(x)).tolist()

        gene_bins[bin_name] = gene_indices
        bin_counter += 1

    return gene_bins

In [18]:
gene_info_df = sdata.tables['table'].var

# Create the gene bins
genes_per_bin = 20  # This is a hyperparameter you can tune
gene_bins = create_gene_bins(gene_info_df, genes_per_bin=genes_per_bin)

# --- You can inspect the first few bins ---
print(f"Created {len(gene_bins)} bins with approximately {genes_per_bin} genes per bin.")
for i, (bin_name, genes) in enumerate(gene_bins.items()):
    if i >= 5:
        break
    print(f"{bin_name}: Contains {len(genes)} genes")

Created 251 bins with approximately 20 genes per bin.
chr1_bin0: Contains 20 genes
chr1_bin1: Contains 20 genes
chr1_bin2: Contains 20 genes
chr1_bin3: Contains 20 genes
chr1_bin4: Contains 20 genes


## Save the Processed Data

Finally, we will save the processed data (gene expression matrix, spatial graph, and gene bins) to files that can be used as input for the ClearCNV model.


In [19]:
# --- Make sure all data is in the correct format ---
# Convert expression matrix to tensor if it's a numpy array
if not isinstance(expression_matrix, torch.Tensor):
    expression_matrix = torch.from_numpy(expression_matrix).float()

# --- Create a dictionary to hold all the data ---
processed_data = {
    'expression_matrix': expression_matrix,
    'spatial_graph': spatial_graph_sparse_tensor,
    'gene_bins': gene_bins
}

# --- Define the output path and save the data ---
output_dir = "../datasets/processed"
output_path = os.path.join(output_dir, "processed_data.pt")

torch.save(processed_data, output_path)

print(f"Processed data saved to: {output_path}")

Processed data saved to: ../datasets/processed/processed_data.pt
